In [1]:
%pip install torch datasets kagglehub tokenizers --quiet

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-lsp-server 1.13.1 requires jedi<0.20.0,>=0.17.2, but you have jedi 0.20.0 which is incompatible.
spyder 6.1.0 requires jedi<0.20.0,>=0.17.2, but you have jedi 0.20.0 which is incompatible.


In [2]:
import re, json, random, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, precision_recall_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [3]:
LABELS = ["toxicity", "hate", "harassment", "abuse"]

JIGSAW_PATH      = "train.csv"
HASOC_TRAIN_PATH = "hindi_dataset.tsv"
HASOC_TEST_PATH  = "hasoc2019_hi_test_gold_2919.tsv"

PRISM_HF_DATASET   = "pankajbiswas6/prism-hinglish-hate-speech"
HATEXPLAIN_HF_DATASET = "Hate-speech-CNERG/hatexplain"
KAGGLE_HINGLISH_SLUG  = "sharduldhekane/code-mixed-hinglish-hate-speech-detection-dataset"

MAX_LEN = 100
VOCAB_SIZE = 20000          
EMBED_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 15
PATIENCE = 3
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

N_SYNTHETIC_HARD_NEGATIVES = 4000

MODEL_PATH = "toxicity_model_v2.pt"
TOKENIZER_PATH = "bpe_tokenizer.json"
THRESHOLDS_PATH = "thresholds.json"
CONFIG_PATH = "model_config_v2.json"


In [4]:
jigsaw_df = pd.read_csv(JIGSAW_PATH)

jigsaw_df["toxicity"]   = ((jigsaw_df["toxic"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)
jigsaw_df["hate"]       = jigsaw_df["identity_hate"].astype(int)
jigsaw_df["harassment"] = ((jigsaw_df["insult"] == 1) | (jigsaw_df["threat"] == 1)).astype(int)
jigsaw_df["abuse"]      = ((jigsaw_df["obscene"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)

jigsaw_df = jigsaw_df.rename(columns={"comment_text": "text"})
jigsaw_df = jigsaw_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
jigsaw_df["label_mask"] = [[1, 1, 1, 1]] * len(jigsaw_df)
jigsaw_df["source"] = "jigsaw_en"

print("jigsaw rows:", len(jigsaw_df))


jigsaw rows: 159571


In [5]:
hasoc_train = pd.read_csv(HASOC_TRAIN_PATH, sep="\t")
hasoc_test  = pd.read_csv(HASOC_TEST_PATH, sep="\t")
hasoc_df = pd.concat([hasoc_train, hasoc_test], ignore_index=True)

hasoc_df["toxicity"] = (hasoc_df["task_1"] == "HOF").astype(int)
hasoc_df["hate"] = 0.0
hasoc_df["harassment"] = 0.0
hasoc_df["abuse"] = 0.0

hasoc_df = hasoc_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
hasoc_df["label_mask"] = [[1, 0, 0, 0]] * len(hasoc_df)
hasoc_df["source"] = "hasoc_hi"

print("hasoc rows:", len(hasoc_df))


hasoc rows: 5982


In [6]:
new_frames = []
try:
    from datasets import load_dataset
    prism = load_dataset(PRISM_HF_DATASET)
    prism_df = pd.concat([prism[split].to_pandas() for split in prism.keys()], ignore_index=True)
    
    text_col = "text" if "text" in prism_df.columns else prism_df.columns[0]
    label_col_candidates = [c for c in prism_df.columns if "label" in c.lower() or "hate" in c.lower()]
    label_col = label_col_candidates[0] if label_col_candidates else None

    prism_clean = pd.DataFrame()
    prism_clean["text"] = prism_df[text_col].astype(str)
    if label_col is not None:
        prism_clean["toxicity"] = prism_df[label_col].astype(int)
    else:
        raise ValueError("Could not auto-detect a label column — inspect prism_df.columns and set label_col manually.")
    prism_clean["hate"] = 0.0
    prism_clean["harassment"] = 0.0
    prism_clean["abuse"] = 0.0
    prism_clean["label_mask"] = [[1, 0, 0, 0]] * len(prism_clean)
    prism_clean["source"] = "prism_hinglish"
    prism_clean = prism_clean.dropna(subset=["text"]).drop_duplicates(subset="text")

    new_frames.append(prism_clean)
    print("prism hinglish rows:", len(prism_clean))
except Exception as e:
    print("Skipped PRISM Hinglish dataset:", e)


README.md:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/3.76M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/621k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17704 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2950 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8852 [00:00<?, ? examples/s]

prism hinglish rows: 29506


In [7]:
try:
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_HINGLISH_SLUG)
    import os
    csvs = [f for f in os.listdir(path) if f.endswith(".csv")]
    assert csvs, "No CSV found in downloaded Kaggle dataset."
    kaggle_df = pd.read_csv(os.path.join(path, csvs[0]))

    text_col = "text" if "text" in kaggle_df.columns else kaggle_df.columns[0]
    label_col_candidates = [c for c in kaggle_df.columns if "label" in c.lower() or "hate" in c.lower() or "class" in c.lower()]
    label_col = label_col_candidates[0] if label_col_candidates else None

    kaggle_clean = pd.DataFrame()
    kaggle_clean["text"] = kaggle_df[text_col].astype(str)
    if label_col is not None:
        raw = kaggle_df[label_col]
        if raw.dtype == object:
            kaggle_clean["toxicity"] = raw.str.lower().isin(["hate", "hateful", "offensive", "1", "true"]).astype(int)
        else:
            kaggle_clean["toxicity"] = (raw > 0).astype(int)
    else:
        raise ValueError("Could not auto-detect a label column — inspect kaggle_df.columns and set label_col manually.")
    kaggle_clean["hate"] = 0.0
    kaggle_clean["harassment"] = 0.0
    kaggle_clean["abuse"] = 0.0
    kaggle_clean["label_mask"] = [[1, 0, 0, 0]] * len(kaggle_clean)
    kaggle_clean["source"] = "kaggle_hinglish"
    kaggle_clean = kaggle_clean.dropna(subset=["text"]).drop_duplicates(subset="text")

    new_frames.append(kaggle_clean)
    print("kaggle hinglish rows:", len(kaggle_clean))
except Exception as e:
    print("Skipped Kaggle Hinglish dataset:", e)

100%|██████████| 2.17M/2.17M [00:03<00:00, 707kB/s]

Extracting files...


kaggle hinglish rows: 29539


In [8]:
try:
    from datasets import load_dataset
    hatexplain = load_dataset(HATEXPLAIN_HF_DATASET)
    hx_rows = []
    for split in hatexplain.keys():
        for row in hatexplain[split]:
            tokens = row["post_tokens"]
            majority_label = max(set(row["annotators"]["label"]), key=row["annotators"]["label"].count)
            # label 0 = hatespeech, 1 = normal, 2 = offensive in this dataset's schema — verify against the dataset card if this looks off
            if majority_label == 1:
                hx_rows.append(" ".join(tokens))
    hx_hard_negatives = pd.DataFrame({"text": hx_rows})
    hx_hard_negatives = hx_hard_negatives.drop_duplicates(subset="text")
    for lbl in LABELS:
        hx_hard_negatives[lbl] = 0
    hx_hard_negatives["label_mask"] = [[1, 1, 1, 1]] * len(hx_hard_negatives)
    hx_hard_negatives["source"] = "hatexplain_normal"
    new_frames.append(hx_hard_negatives)
    print("hatexplain-mined normal rows:", len(hx_hard_negatives))
except Exception as e:
    print("Skipped HateXplain mining:", e)


README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

hatexplain.py:   0%|          | 0.00/4.78k [00:00<?, ?B/s]

Skipped HateXplain mining: Dataset scripts are no longer supported, but found hatexplain.py


In [9]:
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
LATIN_RE = re.compile(r"[a-zA-Z]")

def script_features(text):
    n = max(len(text), 1)
    n_deva = len(DEVANAGARI_RE.findall(text))
    n_latin = len(LATIN_RE.findall(text))
    deva_frac = n_deva / n
    latin_frac = n_latin / n
    is_code_mixed = float(n_deva > 0 and n_latin > 0)
    return [deva_frac, latin_frac, is_code_mixed]


In [10]:
def mine_toxic_words(df, label="toxicity", top_k=150, min_count=20):
    pos_texts = df.loc[df[label] == 1, "text"].str.lower().str.split()
    neg_texts = df.loc[df[label] == 0, "text"].str.lower().str.split()
    pos_counts = Counter(w for toks in pos_texts for w in toks)
    neg_counts = Counter(w for toks in neg_texts for w in toks)
    scored = []
    for w, c in pos_counts.items():
        if c < min_count or not w.isalpha() or len(w) < 3:
            continue
        ratio = c / (neg_counts.get(w, 0) + 1)
        scored.append((ratio, w))
    scored.sort(reverse=True)
    return [w for _, w in scored[:top_k]]

NEGATION_TEMPLATES = [
    "I don't think you're {w} at all.",
    "That's not {w}, honestly.",
    "You're absolutely not {w}.",
    "No one would call that {w}.",
    "I wouldn't say this is {w}.",
    "Some people think this is {w}, but I disagree.",
    "This used to be considered {w}, but not anymore.",
    "Not {w}, just direct.",
]

def generate_hard_negatives(df, n=N_SYNTHETIC_HARD_NEGATIVES, seed=SEED):
    rng = random.Random(seed)
    words = mine_toxic_words(df)
    if not words:
        return pd.DataFrame(columns=["text"] + LABELS + ["label_mask", "source"])
    rows = []
    for _ in range(n):
        w = rng.choice(words)
        t = rng.choice(NEGATION_TEMPLATES).format(w=w)
        rows.append(t)
    out = pd.DataFrame({"text": rows}).drop_duplicates(subset="text")
    for lbl in LABELS:
        out[lbl] = 0
    out["label_mask"] = [[1, 1, 1, 1]] * len(out)
    out["source"] = "synthetic_hard_negative"
    return out

synthetic_df = generate_hard_negatives(jigsaw_df)
print("synthetic hard negatives:", len(synthetic_df))
synthetic_df.head()


synthetic hard negatives: 1159


,text,toxicity,hate,harassment,abuse,label_mask,source
0,I don't think you're haahhahahah at all.,0,0,0,0,"[1, 1, 1, 1]",synthetic_hard_negative
1,No one would call that heil.,0,0,0,0,"[1, 1, 1, 1]",synthetic_hard_negative
2,You're absolutely not fag.,0,0,0,0,"[1, 1, 1, 1]",synthetic_hard_negative
3,"That's not dickhead, honestly.",0,0,0,0,"[1, 1, 1, 1]",synthetic_hard_negative
4,I don't think you're fuckk at all.,0,0,0,0,"[1, 1, 1, 1]",synthetic_hard_negative


In [11]:
combined_df = pd.concat([jigsaw_df, hasoc_df, synthetic_df] + new_frames, ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("combined rows:", len(combined_df))
print(combined_df["source"].value_counts())
print(combined_df[LABELS].mean())

combined rows: 225757
source
jigsaw_en                  159571
kaggle_hinglish             29539
prism_hinglish              29506
hasoc_hi                     5982
synthetic_hard_negative      1159
Name: count, dtype: int64
toxicity      0.202855
hate          0.006224
harassment    0.035649
abuse         0.037771
dtype: float64


In [12]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")

def clean_text(text):
    text = URL_RE.sub(" <URL> ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

combined_df["clean_text"] = combined_df["text"].astype(str).apply(clean_text)
combined_df["script_feats"] = combined_df["clean_text"].apply(script_features)
combined_df[["clean_text", "script_feats", "source"]].head()


,clean_text,script_feats,source
0,""" To those of you who actually know about this...","[0.0, 0.7222222222222222, 0.0]",jigsaw_en
1,"apologies, i didnt notice i actually added the...","[0.0, 0.7859778597785978, 0.0]",jigsaw_en
2,GOOD RIDDANCE. GET LOST.,"[0.0, 0.7916666666666666, 0.0]",jigsaw_en
3,केवल बांग्लादेशी ही इतने बेवकफ हो सकते हैं,"[0.8333333333333334, 0.0, 0.0]",kaggle_hinglish
4,I wouldn't say this is fack.,"[0.0, 0.75, 0.0]",synthetic_hard_negative


In [13]:
train_df, temp_df = train_test_split(combined_df, train_size=0.70, random_state=SEED)
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=SEED)
len(train_df), len(val_df), len(test_df)

(158029, 33864, 33864)

In [14]:
from tokenizers import ByteLevelBPETokenizer

with open("bpe_train_corpus.txt", "w", encoding="utf-8") as f:
    for t in train_df["clean_text"]:
        f.write(t.replace("\n", " ") + "\n")

bpe = ByteLevelBPETokenizer()
bpe.train(
    files=["bpe_train_corpus.txt"],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<URL>"],
)
bpe.save(TOKENIZER_PATH)

PAD_ID = bpe.token_to_id("<PAD>")
UNK_ID = bpe.token_to_id("<UNK>")

def encode(text, max_len=MAX_LEN):
    ids = bpe.encode(text).ids[:max_len]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids

print("BPE vocab size:", bpe.get_vocab_size())
print("sample encode:", bpe.encode("you're not an idiot yaar").tokens)


BPE vocab size: 20000
sample encode: ['you', "'re", 'Ġnot', 'Ġan', 'Ġidiot', 'Ġyaar']


In [15]:
class ToxicityDataset(Dataset):
    def __init__(self, dframe):
        self.texts = dframe["clean_text"].tolist()
        self.labels = dframe[LABELS].values.astype("float32")
        self.masks = np.stack(dframe["label_mask"].values).astype("float32")
        self.script_feats = np.stack(dframe["script_feats"].values).astype("float32")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.script_feats[idx]),
            torch.tensor(self.labels[idx]),
            torch.tensor(self.masks[idx]),
        )

train_ds = ToxicityDataset(train_df)
val_ds   = ToxicityDataset(val_df)
test_ds  = ToxicityDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [16]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, pad_mask):
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(pad_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        return context, weights


class BiLSTMAttnToxicityClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, n_script_feats=3, dropout=0.3, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2 + n_script_feats, num_labels)

    def forward(self, x, script_feats, return_attention=False):
        pad_mask = (x != self.pad_idx).float()
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        context, attn_weights = self.attention(lstm_out, pad_mask)
        context = self.dropout(context)
        fused = torch.cat([context, script_feats], dim=1)
        logits = self.fc(fused)
        if return_attention:
            return logits, attn_weights
        return logits

model = BiLSTMAttnToxicityClassifier(bpe.get_vocab_size(), EMBED_DIM, HIDDEN_DIM, len(LABELS), pad_idx=PAD_ID).to(device)
model


BiLSTMAttnToxicityClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (attention): Attention(
    (attn): Linear(in_features=256, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=259, out_features=4, bias=True)
)

In [17]:
def masked_focal_loss(logits, targets, mask, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    probs = torch.sigmoid(logits)
    ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t).pow(gamma) * ce
    loss = loss * mask
    return loss.sum() / mask.sum().clamp(min=1.0)


In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, sf, y, m in train_loader:
        x, sf, y, m = x.to(device), sf.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        loss = masked_focal_loss(model(x, sf), y, m)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, sf, y, m in val_loader:
            x, sf, y, m = x.to(device), sf.to(device), y.to(device), m.to(device)
            val_loss += masked_focal_loss(model(x, sf), y, m).item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load(MODEL_PATH))


epoch 1: train_loss=0.0152 val_loss=0.0124
epoch 2: train_loss=0.0110 val_loss=0.0113
epoch 3: train_loss=0.0090 val_loss=0.0112
epoch 4: train_loss=0.0072 val_loss=0.0115
epoch 5: train_loss=0.0057 val_loss=0.0131
epoch 6: train_loss=0.0047 val_loss=0.0138
Early stopping.


<All keys matched successfully>

In [19]:
def get_probs(loader):
    model.eval()
    all_probs, all_true, all_masks = [], [], []
    with torch.no_grad():
        for x, sf, y, m in loader:
            x, sf = x.to(device), sf.to(device)
            probs = torch.sigmoid(model(x, sf)).cpu()
            all_probs.append(probs)
            all_true.append(y)
            all_masks.append(m)
    return torch.cat(all_probs).numpy(), torch.cat(all_true).numpy(), torch.cat(all_masks).numpy()

val_probs, val_true, val_mask = get_probs(val_loader)

thresholds = {}
for i, label in enumerate(LABELS):
    valid = val_mask[:, i] == 1
    p, r, t = precision_recall_curve(val_true[valid, i], val_probs[valid, i])
    f1 = 2 * p * r / (p + r + 1e-9)
    best_idx = f1[:-1].argmax() if len(t) > 0 else None
    thresholds[label] = float(t[best_idx]) if best_idx is not None else 0.5

with open(THRESHOLDS_PATH, "w") as f:
    json.dump(thresholds, f)

thresholds

{'toxicity': 0.35584837198257446,
 'hate': 0.3133375644683838,
 'harassment': 0.33578044176101685,
 'abuse': 0.31384769082069397}

In [20]:
test_probs, test_true, test_mask = get_probs(test_loader)

for i, label in enumerate(LABELS):
    valid = test_mask[:, i] == 1
    preds = (test_probs[valid, i] >= thresholds[label]).astype(int)
    p, r, f, _ = precision_recall_fscore_support(test_true[valid, i], preds, average="binary", zero_division=0)
    cm = confusion_matrix(test_true[valid, i], preds)
    fn = int(cm[1][0]) if cm.shape == (2, 2) else None
    print(f"{label:12s} n={valid.sum():>7d} thr={thresholds[label]:.2f} precision={p:.3f} recall={r:.3f} f1={f:.3f} false_negatives={fn}")


toxicity     n=  33864 thr=0.36 precision=0.731 recall=0.816 f1=0.771 false_negatives=1257
hate         n=  24167 thr=0.31 precision=0.359 recall=0.255 f1=0.298 false_negatives=152
harassment   n=  24167 thr=0.34 precision=0.724 recall=0.718 f1=0.721 false_negatives=337
abuse        n=  24167 thr=0.31 precision=0.804 recall=0.789 f1=0.796 false_negatives=269


In [23]:
def predict(text, top_n=5):
    ids = encode(clean_text(text))
    sf = script_features(clean_text(text))
    x = torch.tensor([ids], dtype=torch.long).to(device)
    sf_t = torch.tensor([sf], dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        logits, attn = model(x, sf_t, return_attention=True)
        probs = torch.sigmoid(logits)[0].cpu().tolist()

    per_label = {}
    for label, p in zip(LABELS, probs):
        per_label[label] = {"prob": p, "flagged": p >= thresholds[label]}

    overall = 1 - np.prod([1 - p for p in probs])

    tokens = bpe.encode(clean_text(text)).tokens[:MAX_LEN]
    weights = attn[0][: len(tokens)].cpu().tolist()
    top_tokens = sorted(zip(tokens, weights), key=lambda x: -x[1])[:top_n]

    return {"per_label": per_label, "overall_toxicity": overall, "top_attended_tokens": top_tokens}


for example in [
    "I really enjoyed this movie.",
    "You are a disgusting idiot.",
    "I don't think you're an idiot.",
    "faggot ur so dum lol",
    "tu bahut bekar insaan hai",
]:
    print(example, "->")
    result = predict(example)
    print("  overall:", round(result["overall_toxicity"], 3))
    print("  per-label:", {k: round(v["prob"], 3) for k, v in result["per_label"].items()})
    print("  top tokens:", result["top_attended_tokens"])
    print()


I really enjoyed this movie. ->
  overall: 0.062
  per-label: {'toxicity': 0.022, 'hate': 0.007, 'harassment': 0.016, 'abuse': 0.018}
  top tokens: [('.', 0.6949747800827026), ('Ġenjoyed', 0.20431070029735565), ('Ġmovie', 0.03473604843020439), ('Ġthis', 0.028583068400621414), ('I', 0.02008424699306488)]

You are a disgusting idiot. ->
  overall: 0.98
  per-label: {'toxicity': 0.882, 'hate': 0.121, 'harassment': 0.708, 'abuse': 0.339}
  top tokens: [('Ġidiot', 0.5290862917900085), ('Ġdisgusting', 0.30734944343566895), ('Ġa', 0.0609038881957531), ('You', 0.04471888765692711), ('Ġare', 0.03401941433548927)]

I don't think you're an idiot. ->
  overall: 0.871
  per-label: {'toxicity': 0.648, 'hate': 0.085, 'harassment': 0.517, 'abuse': 0.167}
  top tokens: [('Ġidiot', 0.7507010698318481), ('.', 0.16357627511024475), ('Ġan', 0.05140461027622223), ('Ġthink', 0.01446120161563158), ('I', 0.007200306281447411)]

faggot ur so dum lol ->
  overall: 0.91
  per-label: {'toxicity': 0.526, 'hate': 0.

In [24]:
with open(CONFIG_PATH, "w") as f:
    json.dump({
        "vocab_size": bpe.get_vocab_size(), "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
        "num_labels": len(LABELS), "dropout": DROPOUT, "max_len": MAX_LEN, "labels": LABELS,
        "focal_alpha": FOCAL_ALPHA, "focal_gamma": FOCAL_GAMMA,
    }, f)

print("Saved:", MODEL_PATH, TOKENIZER_PATH, THRESHOLDS_PATH, CONFIG_PATH)


Saved: toxicity_model_v2.pt bpe_tokenizer.json thresholds.json model_config_v2.json


In [25]:
def get_toxicity_level(toxicity_score):

    if toxicity_score < 0.60:
        return {
            "level": "safe",
            "blur": 0,
            "message": "No blur required."
        }

    elif toxicity_score < 0.85:
        return {
            "level": "moderate",
            "blur": 1,
            "message": "Content temporarily blurred due to potentially harmful language."
        }

    else:
        return {
            "level": "high",
            "blur": 2,
            "message": "Content heavily blurred due to highly toxic or harmful language."
        }

In [26]:
def get_toxicity_warning(toxicity_score):

    if toxicity_score < 0.60:

        return {
            "show_warning": False,
            "title": "Content appears safe",
            "reason": "The detected toxicity level is below the warning threshold."
        }

    elif toxicity_score < 0.85:

        return {
            "show_warning": True,
            "title": "Potentially harmful content",
            "reason": (
                "This content may contain offensive, abusive, "
                "or inappropriate language. It has been temporarily blurred."
            )
        }

    else:

        return {
            "show_warning": True,
            "title": "Highly toxic content",
            "reason": (
                "This content has a high probability of containing "
                "severely offensive, abusive, threatening, or harmful language. "
                "It has been heavily blurred for safety."
            )
        }

In [27]:
def get_age_rating(toxicity_score):

    if toxicity_score < 0.60:
        return {
            "age_rating": "13-15",
            "level": "Low"
        }

    elif toxicity_score < 0.70:
        return {
            "age_rating": "15-18",
            "level": "Moderate"
        }

    elif toxicity_score < 0.85:
        return {
            "age_rating": "18-21",
            "level": "High"
        }

    else:
        return {
            "age_rating": "21+",
            "level": "Very High"
        }

In [28]:
toxicity_score = 0.78

threshold = get_toxicity_level(toxicity_score)

warning = get_toxicity_warning(toxicity_score)

age = get_age_rating(toxicity_score)


print("Toxicity Score:", toxicity_score)

print("Toxicity Level:", threshold["level"])

print("Blur Level:", threshold["blur"])

print("Warning:", warning["title"])

print("Reason:", warning["reason"])

print("Age Rating:", age["age_rating"])

Toxicity Score: 0.78
Toxicity Level: moderate
Blur Level: 1
Reason: This content may contain offensive, abusive, or inappropriate language. It has been temporarily blurred.
Age Rating: 18-21
